# 01 – Sanity checks and experiments

Quick sanity checks and experiments for the NVIDIA RAG API: loading the PDF, chunking, embeddings, retrieval, API health, and manual eval against test questions.

## Notebook description

This notebook is **not part of the production API**, but a scratchpad to:
- Verify that PDF loading, chunking, and embeddings behave as expected.
- Sanity‑check retrieval and the `/ask` endpoint end‑to‑end.
- Run a small manual evaluation set and inspect answers/contexts for quality and grounding.

## Contents

1. **Environment & imports** – add project root to `sys.path`, load config and helpers.
2. **PDF loading checks** – confirm file paths and inspect first page.
3. **Chunking checks** – examine chunk counts, IDs, and typical lengths.
4. **Embedding vectorization** – generate a small batch of embeddings and inspect shape/range.
5. **Retrieval sanity checks** – retrieve top‑k chunks for a sample revenue question.
6. **API checks** – spin up `uvicorn`, hit `/health` and `/ask`.
7. **Manual evaluation vs test questions** – load JSON eval set, call `/ask`, and save results to CSV.
8. **Answer and context comparison** – visually compare expected vs model answers and contexts; record conclusions.

## 1. Environment & imports

Set up Python path, working directory, and import config, ingestion, and RAG pipeline helpers used in the checks below.

In [1]:
import os, sys
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
os.chdir('..') 
    
import numpy as np
import pandas as pd
from pathlib import Path
from chromadb import PersistentClient
import requests
import subprocess
import time
import json

from app.config import get_settings
from ingestion.loaders import load_pdf_pages, chunk_pages, Chunk
from ingestion.build_index import embedd_chunks, build_index
from app.rag_pipeline import RetrievedChunk, retrieve

In [2]:
settings = get_settings()
client = PersistentClient(path=settings.index_dir)
collection = client.get_collection(settings.collection_name)

In [3]:
get_settings().openai_api_key[:8] + "..."

'sk-proj-...'

In [4]:
DATA_PATH = Path("./data/NVIDIAAn.pdf")
TEST_FILE_PATH = Path("./tests/data/test_questions.json")
TEST_RESULT_PATH = Path("./data/rag_eval_results.csv")
DATA_PATH, TEST_FILE_PATH, TEST_RESULT_PATH

(PosixPath('data/NVIDIAAn.pdf'),
 PosixPath('tests/data/test_questions.json'),
 PosixPath('data/rag_eval_results.csv'))

## 2. PDF loading checks

Sanity check that the NVIDIA PDF file can be loaded and the first page text looks reasonable.

In [5]:
%%time
pages = list(load_pdf_pages(path=DATA_PATH))
first_page = pages[0]
print(f"first page nr:{first_page.page_number}.")
print(f"Initial text:\n{first_page.text[:200]}.")

first page nr:1.
Initial text:
NVIDIA Announces Financial Results for Second Quarter
Fiscal 2024
Record revenue of $13.51 billion, up 88% from Q1, up 101% from year ago
Record Data Center revenue of $10.32 billion, up 141% from Q1,.
CPU times: user 1.77 s, sys: 7.73 ms, total: 1.78 s
Wall time: 1.78 s


## 3. Chunking checks

Inspect chunk counts, IDs, and lengths produced by `chunk_pages` for the loaded PDF pages.

In [6]:
%%time
chunks = list(chunk_pages(pages))
assert 250 < chunks[0].length <= 800, f"Bad length: {chunks[0].length}"

CPU times: user 102 μs, sys: 23 μs, total: 125 μs
Wall time: 132 μs


In [7]:
print(f"{len(pages)} pages and {len(chunks)} chunks")
print(f"sampel ID {chunks[-1].id}")
print(f"avg chunk len = {int(sum(c.length for c in chunks) / len(chunks))} chars")
print(f"Example chunk: {chunks[-1].text[:50]}...")
# assert all(c.page in range(1, len(pages)+1) for c in chunks)

9 pages and 45 chunks
sampel ID p9_c7
avg chunk len = 709 chars
Example chunk:  other
countries. Other company and product names ...


In [8]:
DATA_PATH.stem

'NVIDIAAn'

## 4. Embedding vectorization

Generate a small batch of embeddings and verify their dimensionality and value ranges.

In [9]:
%%time
embeddings = embedd_chunks(chunks[:2])

CPU times: user 433 ms, sys: 20.9 ms, total: 454 ms
Wall time: 1.73 s


In [10]:
print(f"There are {len(embeddings)} embeddings")
print(
    f"Size of embedding for {get_settings().openai_embedding_model} model is {len(embeddings[0])} dimensions"
)  # Should be 1536
print(f"Embedding values range from {min(embeddings[0]):.3f} to {max(embeddings[0]):.3f}")
print(f"Example embedding: {[round(emb, 3) for emb in embeddings[0][:5]]}...")

There are 2 embeddings
Size of embedding for text-embedding-3-small model is 1536 dimensions
Embedding values range from -0.087 to 0.083
Example embedding: [0.028, -0.03, 0.037, 0.004, 0.034]...


## 5. Retrieval sanity checks

Retrieve top-k chunks for a sample revenue question and check scores, pages, and snippets.

In [11]:
%%time
question = "What was NVIDIA's total revenue for Q2 FY2024?"
top_k = 3

chunks_search = retrieve(question, top_k=top_k)
print(f"Found {len(chunks_search)} chunks:")
for i, chunk in enumerate(chunks_search, 1):
    print(f"\n{i}. Score: {chunk.score:.3f} in Page: {chunk.page}")
    print(f"   ID: {chunk.id}")
    print(f"   Text: {chunk.text[:50]}...")

Found 3 chunks:

1. Score: 0.778 in Page: 1
   ID: p1_c1
   Text: NVIDIA Announces Financial Results for Second Quar...

2. Score: 0.777 in Page: 2
   ID: p2_c1
   Text: NVIDIA’s outlook for the third quarter of fiscal 2...

3. Score: 0.752 in Page: 1
   ID: p1_c3
   Text: .38 billion to shareholders in the form of 7.5 mil...
CPU times: user 39.8 ms, sys: 10.3 ms, total: 50.1 ms
Wall time: 510 ms


## 6. API checks

### 6.1 Health

In [12]:
port = str(8001)
ip = "127.0.0.1"
proc = subprocess.Popen([
    "uvicorn", "app.main:app", 
    "--host", ip, 
    "--port", port
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)  # on 3 seconds fnext cell fails - not enough for uvicorn load
api_url = f"http://{ip}:{port}"

In [13]:
response = requests.get(f"{api_url}/health")
response.json()

{'status': 'ok'}

### 6.2 RAG pipeline

In [14]:
%%time
response = requests.post(
    f"{api_url}/ask",
    json={"question": "What was NVIDIA's total revenue for Q2 FY2024?"}
)
print(response.json()["answer"])
response

NVIDIA's total revenue for Q2 FY2024 was $13.51 billion (Page 1).
CPU times: user 4.19 ms, sys: 1.81 ms, total: 6 ms
Wall time: 3.31 s


<Response [200]>

In [15]:
# proc.terminate()
# proc.kill()

## 7. Manual evaluation vs test questions

In [16]:
with open(TEST_FILE_PATH) as f:
    questions = json.load(f)["eval_questions"]
f"Found {len(questions)} questions"

'Found 5 questions'

In [17]:
%%time
api_answers = []

for question in questions:
    answer = requests.post(f"{api_url}/ask", json={"question": question["question"]})
    if answer.status_code == 200:
        data = answer.json()
        
        api_answers.append({
            "id": question["id"],
            "question": question["question"],
            "expected_answer": question["expected_answer"],
            "model_answer": data["answer"],
            "expected_contexts": question["expected_contexts"],
            "retrieved_contexts": [c["text"][:100] + "..." for c in data["contexts"]],
            "n_contexts": len(data["contexts"])
        })
    else:
        api_answers.append({
            "id": question["id"],
            "question": question["question"],
            "expected_answer": question["expected_answer"],
            "model_answer": f"ERROR: {answer.status_code}",
            "expected_contexts": [],
            "retrieved_contexts": [],
            "n_contexts": 0
        })
        
df_compare_responses = pd.DataFrame(api_answers)
f"Responded to {len(api_answers)} questions"

CPU times: user 15.6 ms, sys: 3.32 ms, total: 18.9 ms
Wall time: 8.4 s


'Responded to 5 questions'

In [18]:
proc.terminate()
proc.kill()

In [19]:
df_compare_responses.to_csv(TEST_RESULT_PATH, index=False)
print(f"Saved datafreame to {TEST_RESULT_PATH}.")

Saved datafreame to data/rag_eval_results.csv.


## 8. Answer and context comparison

### 8.1 Answer comparison

In [20]:
pd.set_option('max_colwidth', 200)
display(df_compare_responses[["id", "question", "expected_answer", "model_answer"]].style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '15px')]}
]))

,id,question,expected_answer,model_answer
0,q1_revenue,What was NVIDIA's total revenue for Q2 FY2024?,$13.51 billion,NVIDIA's total revenue for Q2 FY2024 was $13.51 billion (Page 1).
1,q2_datacenter,What drove NVIDIA's record Data Center revenue?,"$10.32 billion, up 171% YoY","NVIDIA's record Data Center revenue of $10.32 billion was driven by the shipping of the NVIDIA GH200 Grace Hopper Superchip for complex AI and HPC workloads, the availability of the NVIDIA L40S GPU designed to accelerate compute-intensive applications, and the introduction of NVIDIA MGX server reference design. Additionally, major cloud service providers announced massive NVIDIA H100 AI infrastructures, and partnerships were formed to bring NVIDIA AI to various industries, fueling demand (Page 1-2)."
2,q3_products,What new products did NVIDIA announce in Q2 FY2024?,"GH200 Grace Hopper, L40S GPU, MGX, Spectrum-X","In Q2 FY2024, NVIDIA announced several new products including the NVIDIA GH200 Grace Hopper Superchip, the NVIDIA L40S GPU, NVIDIA AI Workbench, NVIDIA AI Enterprise 4.0, NVIDIA MGX server reference design, NVIDIA Spectrum-X networking platform, and the GeForce RTX 4060 family of GPUs. They also introduced NVIDIA Avatar Cloud Engine for Games and new NVIDIA RTX workstations with up to four NVIDIA RTX 6000 Ada GPUs (pages 1-2, 9)."
3,q4_margin,What was NVIDIA's GAAP gross margin for Q2 FY2024?,70.1%,NVIDIA's GAAP gross margin for Q2 FY2024 was 70.1% (Page 1).
4,q5_competitors,Does this document mention AMD or Intel as competitors?,No mention,I don't know. The provided document context does not mention AMD or Intel as competitors.


### 8.2 Context comparison

**Conclusions**:
- Very good ovearll
- "I don't know" works
- Natural language answers are concise and accurate
- Model correctly extracts numbers even from suboptimal contexts (with initial chunk_size: int = 800, overlap: int = 150 ant top_k=5)

In [21]:
print("\n" + "="*110)
print("CONTEXTS COMPARISON")
print("="*110)

for answer in api_answers:
    print(f"\nQ: {answer['id']}")
    print(f"Expected contexts: {answer['expected_contexts']}")
    print("Retrieved contexts:")
    for context in answer['retrieved_contexts']:
        print("->", ' '.join(context.replace("\n", " ").split()))
    print("-" * 110)


CONTEXTS COMPARISON

Q: q1_revenue
Expected contexts: ['13,507', '13.51 billion', '$13.51 billion']
Retrieved contexts:
-> NVIDIA Announces Financial Results for Second Quarter Fiscal 2024 Record revenue of $13.51 billion, ...
-> NVIDIA’s outlook for the third quarter of fiscal 2024 is as follows: Revenue is expected to be $16.0...
-> .38 billion to shareholders in the form of 7.5 million shares repurchased for $3.28 billion, and cas...
-> s begun. Companies worldwide are transitioning from general-purpose to accelerated computing and gen...
-> ets. NVIDIA believes the presentation of its non-GAAP financial measures enhances the user’s overall...
--------------------------------------------------------------------------------------------------------------

Q: q2_datacenter
Expected contexts: ['10.32 billion', 'Data Center', 'H100']
Retrieved contexts:
-> progress since its previous earnings announcement in these areas: Data Center Second-quarter reven...
-> NVIDIA Announces Financial 

**Conclusions**:
- Current context performances **suboptimal** ( With initial chunk_size: int = 800, overlap: int = 150 and top_k=5):
    - q1 revenue: **Good** First chunk has "\$13.51 billion"
    - q4 margin: **Good** Second chunk has "GAAP gross margin 71.5%" (close enough, even if Q3 forecast)
    - q5 competitors: **Good** Correctly returns generic chunks (no competitors mentioned)
    - q2 datacenter: **BAD** Has "Data Center" but misses "$10.32 billion" number and "H100"
    - q3 products: **BAD** Miss - no "GH200", "L40S", "MGX", or "Spectrum-X"
    - **recommendation** Try smaller chunk_size = 40 and overlap = 100
- Attempted solutions:
    - Tested chunk sizes: 400/100 → 800/150 (larger better)
    - Increased top_k: 3 → 5 → 8 (MGX now consistently retrieved)
- **PDF table parsing failure** - `pypdf` misses key facts in tables/nested text:
    - Root cause: "\$10.32B", "GH200", etc. exist but aren't extracted properly
- **Next steps**:
    - Replace `pypdf` with table-aware parsers `unstructured[pdf]` or `PyMuPDF()`, but note:
      - Risk of breaking a working ingestion path and changing chunking behavior, which would require extra debugging and test updates.
      - Additional dependency and environment complexity (native libs, install issues), especially on non-standard setups.
      - Extra implementation and validation time that’s hard to justify for this small showcase, given current answers are already good enough for the task.